# Day 3: Independent Lab — Build Your Own RAG Assistant

## Overview

Build a domain-specific RAG assistant by choosing one of four business tracks. You will design a knowledge base, implement chunking, write a grounding prompt, create a golden test set, and iterate from v1 → v2 with measured improvement.

### Deliverables
| # | Item | Requirement |
|---|------|-------------|
| 1 | Track selection | Clearly marked |
| 2 | Knowledge base | 4+ documents |
| 3 | Chunking strategy | Documented rationale |
| 4 | System prompt v1 | Grounding rules |
| 5 | Golden test set | 10+ questions |
| 6 | v1 evaluation | Accuracy + scores |
| 7 | System prompt v2 | Improved rules |
| 8 | v2 evaluation | Comparison |
| 9 | Error analysis | Patterns + insights |

---
## Setup

In [ ]:
!pip install -q -U google-genai

In [ ]:
# ── Imports ──────────────────────────────────────────────
import os, time, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

from google import genai
from google.genai import types

# ── API Key ───────────────────────────────────────────────
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

MODEL_ID = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"

print(f"API key loaded: {'yes' if API_KEY else 'no'}")
print(f"Generation model: {MODEL_ID}")
print(f"Embedding model:  {EMBEDDING_MODEL}")

In [ ]:
# ── Infrastructure (same as Guided Lab) ──────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, temperature=0.7, max_tokens=1000, log=True, label=None):
    """Generate free-form text. Returns raw string."""
    t0 = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"temperature": temperature, "max_output_tokens": max_tokens},
    )
    latency = time.time() - t0
    text = response.text
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(), "label": label or "generate",
            "type": "free_form",
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": text[:300] + "..." if len(text) > 300 else text,
            "response_length": len(text),
            "latency_s": round(latency, 2),
        })
    return text

def generate_structured(prompt, schema_model, temperature=0.2, log=True, label=None):
    """Generate structured JSON output. Returns Pydantic model instance."""
    t0 = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_schema": schema_model,
        },
    )
    latency = time.time() - t0
    result = schema_model.model_validate_json(response.text)
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(), "label": label or "structured",
            "type": "structured",
            "schema": schema_model.__name__,
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": response.text[:300],
            "response_length": len(response.text),
            "latency_s": round(latency, 2),
        })
    return result

def show_log():
    """Display prompt log as DataFrame."""
    if not PROMPT_LOG:
        print("No API calls logged yet.")
        return None
    return pd.DataFrame(PROMPT_LOG)

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed one or more texts using Gemini Embeddings API."""
    if isinstance(texts, str):
        texts = [texts]
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return [np.array(e.values) for e in response.embeddings]

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def search(query, doc_embeddings, documents, top_k=3):
    """Find top-k most similar documents to a query."""
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    scores = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_emb, doc_emb)
        scores.append((i, sim, documents[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

def rag_query(question, documents, doc_embeddings, top_k=3, system_prompt=None):
    """Complete RAG pipeline: retrieve → augment → generate."""
    results = search(question, doc_embeddings, documents, top_k)
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{doc}" for i, (_, _, doc) in enumerate(results)
    )
    if system_prompt is None:
        system_prompt = (
            "You are a helpful assistant. Answer based ONLY on the provided context. "
            "If the answer is not in the context, say \"I don't have that information.\""
        )
    prompt = f"""{system_prompt}

<context>
{context}
</context>

Question: {question}"""
    answer = generate(prompt, temperature=0.3, label=f"rag_{question[:30]}")
    return answer, results

def chunk_sentences(text, max_words=60, overlap_words=15):
    """Split text at sentence boundaries with overlap."""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('. ') if s.strip()]
    chunks = []
    current = []
    current_len = 0
    for sent in sentences:
        sent_words = len(sent.split())
        if current_len + sent_words > max_words and current:
            chunks.append(". ".join(current) + ".")
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                if overlap_len + len(s.split()) <= overlap_words:
                    overlap_sents.insert(0, s)
                    overlap_len += len(s.split())
                else:
                    break
            current = overlap_sents
            current_len = overlap_len
        current.append(sent)
        current_len += sent_words
    if current:
        chunks.append(". ".join(current) + ".")
    return chunks

# ── Additional helpers for this lab ──────────────────────

def evaluate_retrieval(golden_set, chunks, chunk_embeddings, system_prompt, top_k=3):
    """Run RAG on a golden set and measure keyword accuracy."""
    results = []
    for qa in golden_set:
        answer, retrieved = rag_query(
            qa["question"], chunks, chunk_embeddings,
            top_k=top_k, system_prompt=system_prompt
        )
        answer_lower = answer.lower()
        keyword_hit = any(kw.lower() in answer_lower for kw in qa["expected_keywords"])
        results.append({
            "id": qa["id"],
            "question": qa["question"],
            "difficulty": qa["difficulty"],
            "answer": answer,
            "keyword_match": keyword_hit,
            "top_score": retrieved[0][1],
        })
    return pd.DataFrame(results)

def compare_versions(v1_df, v2_df):
    """Compare two evaluation DataFrames."""
    v1_acc = v1_df["keyword_match"].mean()
    v2_acc = v2_df["keyword_match"].mean()
    print(f"v1 accuracy: {v1_acc:.0%} ({v1_df['keyword_match'].sum()}/{len(v1_df)})")
    print(f"v2 accuracy: {v2_acc:.0%} ({v2_df['keyword_match'].sum()}/{len(v2_df)})")
    print(f"Improvement: {v2_acc - v1_acc:+.0%}")
    # Show per-question comparison
    comparison = v1_df[["id", "difficulty", "keyword_match"]].rename(
        columns={"keyword_match": "v1_correct"}
    )
    comparison["v2_correct"] = v2_df["keyword_match"].values
    comparison["changed"] = comparison["v1_correct"] != comparison["v2_correct"]
    print("\nPer-question comparison:")
    print(comparison.to_string(index=False))
    return comparison

# ── Hybrid Retrieval (TF-IDF + Semantic) ─────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

def build_sparse_index(chunks):
    """Build a TF-IDF sparse index for keyword retrieval."""
    vectorizer = TfidfVectorizer(stop_words='english')
    matrix = vectorizer.fit_transform(chunks)
    return vectorizer, matrix

def search_sparse(query, vectorizer, tfidf_matrix, chunks, top_k=5):
    """Sparse keyword search using TF-IDF cosine similarity."""
    from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
    query_vec = vectorizer.transform([query])
    sims = sklearn_cosine(query_vec, tfidf_matrix).ravel()
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i]), chunks[i]) for i in top_indices if sims[i] > 0]

def hybrid_search(query, doc_embeddings, chunks, vectorizer, tfidf_matrix,
                  top_k=5, semantic_weight=0.6, keyword_weight=0.4):
    """
    Combine semantic and keyword search results.
    Merges by chunk index, deduplicates, and scores with weighted average.
    """
    # Semantic search
    sem_results = search(query, doc_embeddings, chunks, top_k=top_k * 2)
    # Sparse search
    sparse_results = search_sparse(query, vectorizer, tfidf_matrix, chunks, top_k=top_k * 2)
    
    # Merge by chunk index
    scores = {}
    for idx, score, text in sem_results:
        scores[idx] = {"semantic": score, "sparse": 0.0, "text": text}
    for idx, score, text in sparse_results:
        if idx in scores:
            scores[idx]["sparse"] = score
        else:
            scores[idx] = {"semantic": 0.0, "sparse": score, "text": text}
    
    # Weighted combination
    combined = []
    for idx, s in scores.items():
        combo = semantic_weight * s["semantic"] + keyword_weight * s["sparse"]
        combined.append((idx, combo, s["semantic"], s["sparse"], s["text"]))
    
    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:top_k]

def search_with_filter(query, doc_embeddings, chunks, chunk_metadata, top_k=3, filter_field=None, filter_value=None):
    """Semantic search with optional metadata filtering."""
    query_emb = embed_texts(query, task_type="RETRIEVAL_QUERY")[0]
    scores = []
    for i, doc_emb in enumerate(doc_embeddings):
        # Apply metadata filter
        if filter_field and filter_value:
            if chunk_metadata[i].get(filter_field) != filter_value:
                continue
        sim = cosine_similarity(query_emb, doc_emb)
        scores.append((i, sim, chunks[i]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# ── LLM Re-ranking (optional) ────────────────────────────

RERANK_PROMPT = """You are re-ranking retrieved excerpts for a question.
Return the {k} most relevant excerpt numbers as a JSON list.

Question: {question}

Excerpts:
{excerpts}

Return ONLY a JSON list of excerpt numbers like: [2, 0, 3]
The numbers refer to the excerpt indices shown above."""

def rerank_with_llm(question, candidates, k=3):
    """Use LLM to re-rank candidates by relevance. Returns re-ordered list."""
    excerpts = "\n\n".join(
        f"[Excerpt {i}]: {text[:200]}..."
        for i, (_, _, text) in enumerate(candidates)
    )
    prompt = RERANK_PROMPT.format(question=question, excerpts=excerpts, k=k)
    response = generate(prompt, temperature=0.1, label="rerank")
    try:
        import json as _json
        indices = _json.loads(response.strip())
        reranked = [candidates[i] for i in indices if i < len(candidates)]
        return reranked[:k]
    except Exception:
        return candidates[:k]  # Fallback to original order

print("All infrastructure ready (includes hybrid retrieval, metadata filtering, and LLM re-ranking).")

---
## Step 1: Choose Your Track

Select one of four tracks. Set `SELECTED_TRACK` below.

In [ ]:
# ── TODO: Set your track ─────────────────────────────────
SELECTED_TRACK = ""  # Set to "A", "B", "C", or "D"
assert SELECTED_TRACK in ("A", "B", "C", "D"), "Please set SELECTED_TRACK to A, B, C, or D"
print(f"Selected track: {SELECTED_TRACK}")

In [ ]:
# ── Track Data ───────────────────────────────────────────

TRACK_DATA = {
    "A": {
        "name": "HR Policy Assistant",
        "documents": [
            # Doc 0: Remote Work Policy
            """Remote Work Policy (Effective January 2025)

Eligibility: All full-time employees who have completed their 90-day probation 
period are eligible for remote work. Contractors and interns must work on-site.

Schedule: Employees may work remotely up to 3 days per week. Core collaboration 
days are Tuesday and Thursday — attendance is mandatory and in-person. Managers 
may require additional on-site days during project-critical phases.

Equipment: The company provides a one-time EUR 500 stipend for home office 
equipment (desk, chair, monitor). Laptops are issued by IT. Internet costs are 
reimbursed up to EUR 30/month with receipt submission.

Performance: Remote workers are evaluated on output, not hours. However, 
employees must be available during core hours (10:00-15:00 CET) for meetings 
and collaboration.""",

            # Doc 1: Leave Policy
            """Annual Leave and Absence Policy (Effective January 2025)

Annual Leave: Full-time employees receive 30 days of paid annual leave per year, 
plus public holidays. Part-time employees receive leave pro-rata.

Carry-Over: Up to 5 unused days may be carried into the next calendar year. 
Carried-over days must be used by March 31 or they expire.

Sick Leave: Employees receive continued pay for up to 6 weeks of illness per 
year. A doctor's note is required from the third consecutive sick day.

Parental Leave: Birth parents receive 14 weeks of fully paid parental leave. 
Non-birth parents receive 4 weeks of fully paid leave. Additional unpaid leave 
of up to 6 months is available upon request.

Special Leave: 2 days for marriage, 2 days for bereavement of immediate family, 
1 day for moving house.""",

            # Doc 2: Benefits Overview
            """Employee Benefits Overview (2025)

Health Insurance: The company covers 50% of statutory health insurance 
contributions. Supplementary private dental insurance is available at group 
rates through our partner Allianz.

Pension: The company matches employee pension contributions up to 4% of gross 
salary through our company pension scheme (bAV).

Professional Development: Each employee has an annual learning budget of 
EUR 1,500 for courses, conferences, and certifications. Unused budget does 
not carry over.

Wellness: EUR 50/month gym or fitness subsidy (receipt required). Annual 
company health check-up offered in Q2.

Other Benefits: Public transport subsidy (Deutschlandticket), free drinks and 
snacks in the office, quarterly team events with EUR 50/person budget.""",

            # Doc 3: Work Hours Policy
            """Working Hours Policy (Effective January 2025)

Standard Hours: Full-time employees work 40 hours per week. Core hours are 
10:00 to 15:00 CET — all employees must be available during this window.

Flextime: Outside of core hours, employees may arrange their schedule 
flexibly. Start times between 07:00 and 10:00 are permitted.

Overtime: Overtime must be pre-approved by the direct manager. Approved 
overtime is compensated as time-off-in-lieu (TOIL) at a 1:1 ratio. 
Overtime exceeding 10 hours per month requires HR approval.

On-Call: Engineers on the on-call rotation receive EUR 200/week flat 
compensation plus EUR 50 per incident responded to outside business hours.

Time Tracking: All employees must log hours in the Personio system by 
end of each Friday. Failure to log for 2+ consecutive weeks will trigger 
a reminder from HR.""",
        ],
    },
    "B": {
        "name": "Product Documentation",
        "documents": [
            # Doc 0: Authentication Guide
            """DataSync Pro — Authentication Guide (v3.2)

API Keys: Every API request must include a valid API key in the 
X-API-Key header. Keys are generated in the Dashboard under Settings > 
API Keys. Each organisation can have up to 10 active keys.

OAuth 2.0: For user-level access, DataSync Pro supports the Authorization 
Code flow. Redirect URI must be registered in advance. Access tokens expire 
after 1 hour; refresh tokens are valid for 30 days.

Token Refresh: POST to /oauth/token with grant_type=refresh_token and your 
refresh_token. The response includes a new access_token and a new refresh_token 
(rotate both).

Rate Limits: API key requests are limited to 100 requests/minute and 
10,000 requests/day. OAuth tokens are limited to 200 requests/minute. 
Exceeding limits returns HTTP 429 with a Retry-After header.""",

            # Doc 1: REST API Reference
            """DataSync Pro — REST API Reference (v3.2)

Base URL: https://api.datasyncpro.com/v3

GET /connections — List all active connections. Returns array of connection 
objects. Supports ?status=active|paused|error filtering.

POST /connections — Create a new connection. Required body: source_type, 
destination_type, schedule. Returns connection_id.

GET /connections/{id}/runs — List sync runs for a connection. Returns 
timestamps, row counts, and status. Supports ?limit and ?offset pagination.

POST /sync/trigger — Manually trigger a sync. Required: connection_id. 
Optional: full_sync (boolean, default false for incremental).

GET /health — System health check. Returns status, version, and uptime. 
No authentication required.

Error Responses: 400 (bad request), 401 (invalid key), 403 (insufficient 
permissions), 404 (resource not found), 429 (rate limited), 500 (server error).""",

            # Doc 2: SDK Quickstart
            """DataSync Pro — SDK Quickstart Guide

Python SDK:
  pip install datasyncpro
  from datasyncpro import Client
  client = Client(api_key="your-key")
  connections = client.connections.list()

JavaScript SDK:
  npm install @datasyncpro/sdk
  import { DataSyncClient } from '@datasyncpro/sdk';
  const client = new DataSyncClient({ apiKey: 'your-key' });
  const connections = await client.connections.list();

Java SDK:
  Add Maven dependency: com.datasyncpro:sdk:3.2.0
  DataSyncClient client = new DataSyncClient("your-key");
  List<Connection> connections = client.connections().list();

All SDKs support automatic retry with exponential backoff for 429 and 5xx 
errors. Default: 3 retries with 1s/2s/4s delays. Configure via 
client.config.maxRetries.""",

            # Doc 3: Common Errors
            """DataSync Pro — Troubleshooting Common Errors

HTTP 401 Unauthorized:
- API key is missing, expired, or revoked. Regenerate in Dashboard.
- OAuth token has expired. Use the refresh token to obtain a new one.

HTTP 403 Forbidden:
- Your API key lacks permission for this endpoint. Check key scopes in Dashboard.
- Organisation-level endpoints require admin role.

HTTP 429 Too Many Requests:
- You've exceeded rate limits. Check the Retry-After header and wait.
- Implement exponential backoff: wait 1s, 2s, 4s between retries.
- Consider batching requests to reduce call volume.

HTTP 500 Internal Server Error:
- Temporary server issue. Retry after 30 seconds.
- If persistent (>5 minutes), check status.datasyncpro.com.
- Contact support@datasyncpro.com with your request_id from the response header.

Sync Failures:
- "Schema mismatch": Source schema changed. Re-map fields in Dashboard.
- "Connection timeout": Destination unreachable. Check firewall rules.
- "Row limit exceeded": Free plan limited to 100K rows per sync.""",
        ],
    },
    "C": {
        "name": "Customer Support",
        "documents": [
            # Doc 0: Account Access
            """CloudBase — Account Access Guide

Password Reset: Click "Forgot Password" on the login page. A reset link 
is sent to your registered email (valid for 24 hours). If you don't receive 
it, check spam/junk folders or contact support.

Two-Factor Authentication (2FA): Enable 2FA in Settings > Security. We 
support authenticator apps (Google Authenticator, Authy) and SMS. Recovery 
codes are provided during setup — store them securely.

Locked Accounts: After 5 failed login attempts, accounts are locked for 
30 minutes. Contact support to unlock immediately if urgent.

Single Sign-On (SSO): Enterprise plans support SAML 2.0 SSO. Configuration 
requires: Identity Provider metadata URL, attribute mapping, and admin 
approval. Setup takes 1-2 business days.""",

            # Doc 1: Billing Guide
            """CloudBase — Billing & Subscription Guide

Plans: Free (5 users, 1GB), Team (EUR 12/user/month, 50GB), Enterprise 
(custom pricing, unlimited storage). All prices exclude VAT.

Upgrades: Upgrades take effect immediately. You are charged a prorated 
amount for the remainder of the billing cycle.

Downgrades: Downgrades take effect at the start of the next billing cycle. 
No partial refunds are issued for the current cycle.

Refund Policy: Full refunds are available within 14 days of initial 
purchase or upgrade. After 14 days, no refunds are issued. To request a 
refund, email billing@cloudbase.io with your account ID.

Payment Methods: Credit card (Visa, Mastercard, Amex), SEPA direct debit 
(EU only), bank transfer (annual plans only, Enterprise tier).

Invoices: Monthly invoices are sent on the 1st of each month to the billing 
email. Past invoices are available in Settings > Billing > Invoice History.""",

            # Doc 2: Troubleshooting
            """CloudBase — Troubleshooting Guide

Slow Performance:
- Clear browser cache and cookies. Try incognito/private mode.
- Check status.cloudbase.io for ongoing incidents.
- Large dashboards (>50 widgets) may load slowly. Consider splitting them.

Sync Errors:
- "Sync conflict": Two users edited the same record. Manually resolve by 
  choosing the correct version in the conflict resolution panel.
- "Connection lost": Check internet connectivity. CloudBase auto-retries 
  every 30 seconds for up to 5 minutes.

Browser Compatibility:
- Supported: Chrome 90+, Firefox 88+, Edge 90+, Safari 15+.
- NOT supported: Internet Explorer (any version).
- Mobile: Use the CloudBase mobile app for iOS 15+ and Android 12+.

Export Issues:
- CSV exports are limited to 100,000 rows. Use the API for larger datasets.
- PDF reports timeout after 60 seconds. Reduce the date range or number of 
  charts to speed up generation.""",

            # Doc 3: Usage Limits
            """CloudBase — Usage Limits & Fair Use Policy

Storage Quotas:
- Free: 1 GB total. Team: 50 GB total (shared across team). 
  Enterprise: Unlimited (fair use applies).
- File upload limit: 100 MB per file on all plans.

API Limits:
- Free: 1,000 API calls/day. Team: 50,000 API calls/day. 
  Enterprise: 500,000 API calls/day (higher limits available on request).
- Rate limit: 60 requests/minute per API key.

Team Size:
- Free: Up to 5 users. Team: Up to 100 users. Enterprise: Unlimited.
- Adding users beyond plan limits requires an upgrade.

Fair Use Policy: "Unlimited" features (Enterprise storage, API calls) are 
subject to fair use. Sustained usage exceeding 10x the median for your plan 
tier may trigger a review. We will contact you before taking any action.

Data Retention: Deleted data is retained in backups for 30 days (Team) or 
90 days (Enterprise). Free plan data is not backed up.""",
        ],
    },
    "D": {
        "name": "Research Analyst",
        "documents": [
            # Doc 0: Market Overview
            """Cloud Analytics Market — Overview (Q4 2024 Report)

Total Addressable Market (TAM): The global cloud analytics market was valued 
at USD 65 billion in 2024 and is projected to reach USD 130 billion by 2028, 
growing at a CAGR of 19%.

Key Segments: Business Intelligence (35% of market), Data Integration (25%), 
Advanced Analytics & ML (25%), Data Governance (15%).

Growth Drivers: Increasing cloud adoption (78% of enterprises now multi-cloud), 
rising data volumes (estimated 120 ZB generated globally in 2024), and 
regulatory compliance requirements (GDPR, CCPA, AI Act).

Market Maturity: The BI segment is mature with consolidation expected. 
Advanced Analytics & ML is the fastest-growing segment at 28% CAGR, driven 
by generative AI adoption. Data Governance is emerging as organisations 
prepare for the EU AI Act (effective August 2025).""",

            # Doc 1: Competitive Landscape
            """Cloud Analytics Market — Competitive Landscape (Q4 2024)

Top 5 Players by Market Share:
1. Snowflake (18%): Dominant in data warehousing. Strengths: performance, 
   ecosystem. Weakness: premium pricing, vendor lock-in concerns.
2. Databricks (15%): Leader in ML/AI workloads. Strengths: open-source 
   (Apache Spark), unified analytics. Weakness: complexity for non-technical users.
3. Microsoft Fabric (14%): Fastest-growing. Strengths: Office 365 integration, 
   enterprise relationships. Weakness: relatively new, fewer third-party integrations.
4. Google BigQuery (12%): Strong in serverless analytics. Strengths: pricing 
   model, Gemini AI integration. Weakness: smaller partner ecosystem.
5. AWS Redshift (11%): Established player. Strengths: AWS ecosystem, 
   broad feature set. Weakness: aging architecture, migration complexity.

Rest of Market (30%): Includes Tableau (Salesforce), Qlik, Palantir, dbt Labs, 
and 200+ niche players. Consolidation is accelerating — 23 acquisitions in 2024.""",

            # Doc 2: Customer Trends
            """Cloud Analytics Market — Customer Trends (Q4 2024)

Adoption Patterns: 67% of enterprises now use 2+ analytics platforms 
(up from 45% in 2022). Multi-tool strategies are driven by best-of-breed 
preferences and avoiding vendor lock-in.

Churn Drivers (in order of impact):
1. Total cost of ownership surprises (cited by 34% of churned customers)
2. Poor data integration with existing tools (28%)
3. Insufficient self-service capabilities for business users (22%)
4. Performance issues at scale (16%)

NPS Benchmarks: Industry average NPS is 32. Leaders: Databricks (52), 
Snowflake (45). Laggards: AWS Redshift (18), legacy on-premise tools (5).

Decision Criteria (ranked by importance): 1) Total cost of ownership, 
2) Ease of integration, 3) Performance at scale, 4) AI/ML capabilities, 
5) Vendor support quality.

Buying Process: Average sales cycle is 4.5 months for enterprise deals. 
Technical evaluation involves 3-4 stakeholders. CFO sign-off required 
for deals exceeding USD 100K ARR.""",

            # Doc 3: Regional Insights
            """Cloud Analytics Market — Regional Insights (Q4 2024)

North America (45% of global market): Most mature market. Cloud-first 
mandates are standard in 80% of enterprises. Average spend: USD 2.1M per 
enterprise. Key trend: Consolidation of analytics tools to reduce costs.

Europe (30%): Growing at 22% CAGR, faster than global average. GDPR has 
made data governance a top priority. EU AI Act (effective August 2025) is 
driving demand for explainability and audit tools. Average spend: EUR 1.4M.
Germany and UK are largest markets (combined 55% of European spend).

Asia-Pacific (20%): Fastest-growing region at 26% CAGR. India and Southeast 
Asia are key growth markets. Government digitalisation programs are a major 
driver. Average spend is lower (USD 0.6M) but growing rapidly.

Rest of World (5%): Nascent markets in Middle East (UAE, Saudi Arabia) and 
Latin America (Brazil, Mexico). Primarily adopting SaaS models. Cloud 
infrastructure availability remains a constraint in some regions.

Pricing Differences: Enterprise pricing is 15-20% lower in APAC and 
emerging markets compared to North America. EU pricing is comparable 
to North America.""",
        ],
    },
}

# Select the correct track data
track = TRACK_DATA[SELECTED_TRACK]
documents = track["documents"]
print(f"Track: {track['name']}")
print(f"Documents loaded: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"  Doc {i}: {doc.split(chr(10))[0][:60]}... ({len(doc.split())} words)")

---
## Step 2: Design Your Chunking Strategy

Choose your chunk size and overlap. Start with the defaults and adjust after evaluating.

In [ ]:
# ── TODO: Configure your chunking parameters ─────────────
CHUNK_MAX_WORDS = 80    # Adjust: try 60-120
CHUNK_OVERLAP_WORDS = 20  # Adjust: try 10-25

# Chunk all documents WITH metadata
all_chunks = []
chunk_metadata = []  # Parallel list of metadata dicts

for doc_idx, doc in enumerate(documents):
    chunks = chunk_sentences(doc, max_words=CHUNK_MAX_WORDS, overlap_words=CHUNK_OVERLAP_WORDS)
    doc_title = doc.strip().split('\n')[0]  # First line as title
    for chunk_idx, chunk_text in enumerate(chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append({
            "chunk_id": f"DOC{doc_idx}::C{chunk_idx+1}",
            "doc_index": doc_idx,
            "doc_title": doc_title,
        })

print(f"Chunking: max_words={CHUNK_MAX_WORDS}, overlap={CHUNK_OVERLAP_WORDS}")
print(f"Total chunks: {len(all_chunks)}\n")
for i, (chunk, meta) in enumerate(zip(all_chunks, chunk_metadata)):
    print(f"Chunk {i:2d} [{meta['chunk_id']}] ({len(chunk.split()):3d} words): {chunk[:70]}...")

In [ ]:
# Embed all chunks (dense index)
chunk_embeddings = embed_texts(all_chunks, task_type="RETRIEVAL_DOCUMENT")
print(f"Dense index: {len(chunk_embeddings)} chunks (dimension: {len(chunk_embeddings[0])})")

# Build sparse index (TF-IDF for keyword search)
tfidf_vectorizer, tfidf_matrix = build_sparse_index(all_chunks)
print(f"Sparse index: {tfidf_matrix.shape[0]} chunks, {tfidf_matrix.shape[1]} vocabulary terms")

---
## Step 2b: Test Hybrid Retrieval

Compare semantic-only vs. hybrid (semantic + keyword) retrieval. Hybrid search catches queries where exact terms matter.

In [ ]:
# Compare semantic-only vs hybrid retrieval
test_query = all_chunks[0].split()[:5]  # Use first few words from doc as test
test_query = " ".join(test_query)

print(f"Test query: \"{test_query}\"\n")

# Semantic only
print("Semantic-only results:")
sem_results = search(test_query, chunk_embeddings, all_chunks, top_k=3)
for idx, score, text in sem_results:
    print(f"  [{score:.3f}] Chunk {idx}: {text[:70]}...")

# Hybrid
print("\nHybrid results (0.6 semantic + 0.4 keyword):")
hyb_results = hybrid_search(
    test_query, chunk_embeddings, all_chunks,
    tfidf_vectorizer, tfidf_matrix, top_k=3
)
for idx, combo, sem, sparse, text in hyb_results:
    print(f"  [{combo:.3f}] Chunk {idx} (sem={sem:.3f}, kw={sparse:.3f}): {text[:60]}...")

print("\nHybrid search boosts chunks that match both meaning AND exact keywords.")

### Step 2c: Metadata Filtering

In production, you often want to restrict search to specific document categories (e.g., only search HR policies, not engineering docs).

In [ ]:
# Demonstrate metadata filtering
print("All documents in index:")
seen = set()
for meta in chunk_metadata:
    if meta["doc_title"] not in seen:
        print(f"  Doc {meta['doc_index']}: {meta['doc_title'][:60]}")
        seen.add(meta["doc_title"])

# Search with filter (restrict to first document only)
print(f"\nFiltered search (doc_index=0 only):")
filtered = search_with_filter(
    "TODO: enter a question here",
    chunk_embeddings, all_chunks, chunk_metadata,
    top_k=3, filter_field="doc_index", filter_value=0
)
for idx, score, text in filtered:
    meta = chunk_metadata[idx]
    print(f"  [{score:.3f}] {meta['chunk_id']}: {text[:70]}...")

print("\nMetadata filtering ensures the retriever only searches relevant document subsets.")

---
## Step 3: Write Your System Prompt (v1)

In [ ]:
# ── TODO: Write your grounding system prompt ─────────────

SYSTEM_PROMPT_V1 = """You are a [ROLE] assistant for [DOMAIN].

Rules:
- Answer ONLY based on the provided context excerpts
- If the answer is not in the context, say: "I don't have that information in the provided documents."
- Cite which chunk(s) support your answer using [Chunk N] notation
- Keep answers concise (under 80 words)
- Do not make up information or combine context with your own knowledge

Scope:
- You answer questions about [TOPICS]
- For questions outside this scope, say: "That's outside my area of expertise."

TODO: Customize the placeholders above for your chosen track!
"""

print("System prompt v1:")
print(SYSTEM_PROMPT_V1)

---
## Step 4: Create Your Golden Test Set

Create 10+ questions BEFORE running the pipeline. Mix easy, medium, hard, and refusal.

In [ ]:
# ── TODO: Create your golden test set ────────────────────
# Include at least: 5-6 easy, 2-3 medium/edge, 2-3 refusal

GOLDEN_SET = [
    # Easy questions (direct answers in a single chunk)
    {
        "id": "Q01",
        "question": "TODO: Write your first easy question",
        "expected_keywords": ["TODO"],
        "difficulty": "easy",
    },
    {
        "id": "Q02",
        "question": "TODO: Write your second easy question",
        "expected_keywords": ["TODO"],
        "difficulty": "easy",
    },
    {
        "id": "Q03",
        "question": "TODO",
        "expected_keywords": ["TODO"],
        "difficulty": "easy",
    },
    {
        "id": "Q04",
        "question": "TODO",
        "expected_keywords": ["TODO"],
        "difficulty": "easy",
    },
    {
        "id": "Q05",
        "question": "TODO",
        "expected_keywords": ["TODO"],
        "difficulty": "easy",
    },
    # Medium questions (may require combining chunks)
    {
        "id": "Q06",
        "question": "TODO: Write a question that spans multiple documents",
        "expected_keywords": ["TODO"],
        "difficulty": "medium",
    },
    {
        "id": "Q07",
        "question": "TODO",
        "expected_keywords": ["TODO"],
        "difficulty": "medium",
    },
    # Edge case questions
    {
        "id": "Q08",
        "question": "TODO: Write an ambiguous or boundary question",
        "expected_keywords": ["TODO"],
        "difficulty": "hard",
    },
    # Refusal questions (no answer in knowledge base)
    {
        "id": "Q09",
        "question": "TODO: Write a question with NO answer in your documents",
        "expected_keywords": ["don't have", "not in", "no information", "outside"],
        "difficulty": "refusal",
    },
    {
        "id": "Q10",
        "question": "TODO: Another out-of-scope question",
        "expected_keywords": ["don't have", "not in", "no information", "outside"],
        "difficulty": "refusal",
    },
]

print(f"Golden set: {len(GOLDEN_SET)} questions")
for q in GOLDEN_SET:
    print(f"  [{q['difficulty']:7s}] {q['id']}: {q['question']}")

---
## Step 5: Evaluate v1

In [ ]:
# Run v1 evaluation
# (Make sure you've filled in your SYSTEM_PROMPT_V1 and GOLDEN_SET above!)

v1_results = evaluate_retrieval(GOLDEN_SET, all_chunks, chunk_embeddings, SYSTEM_PROMPT_V1)

v1_acc = v1_results["keyword_match"].mean()
print(f"\nv1 Keyword Accuracy: {v1_acc:.0%} ({v1_results['keyword_match'].sum()}/{len(v1_results)})\n")
print(v1_results[["id", "difficulty", "keyword_match", "top_score"]].to_string(index=False))

In [ ]:
# LLM-as-Judge scoring for v1
class RAGScore(BaseModel):
    groundedness: int = Field(description="1-5: Is the answer supported by the context?")
    relevance: int = Field(description="1-5: Does the answer address the question?")
    explanation: str = Field(description="Brief explanation of the scores")

v1_scores = []
for _, row in v1_results.iterrows():
    eval_prompt = f"""Rate this RAG system answer.

Question: {row['question']}
Answer: {row['answer']}

Score groundedness (1-5): Is the answer based on facts from context, not made up?
Score relevance (1-5): Does the answer address the question asked?"""
    score = generate_structured(eval_prompt, RAGScore, label=f"eval_v1_{row['id']}")
    v1_scores.append({"id": row["id"], "groundedness": score.groundedness,
                       "relevance": score.relevance, "explanation": score.explanation})

v1_scores_df = pd.DataFrame(v1_scores)
print("v1 LLM-as-Judge Scores:")
print(v1_scores_df[["id", "groundedness", "relevance"]].to_string(index=False))
print(f"\nAvg Groundedness: {v1_scores_df['groundedness'].mean():.1f}/5")
print(f"Avg Relevance:    {v1_scores_df['relevance'].mean():.1f}/5")

---
## Step 6: Improve Your Prompt → v2

Based on v1 errors, improve your system prompt. Common improvements:
- Add specific rules for categories or edge cases
- Add examples of good answers
- Strengthen refusal instructions
- Add format requirements (citations, length)

In [ ]:
# ── TODO: Write your improved system prompt ──────────────

SYSTEM_PROMPT_V2 = """TODO: Write your improved system prompt here.

Start from v1 and add:
1. Specific rules based on errors you observed
2. Examples of good answers for tricky questions
3. Stronger refusal instructions
4. Any additional scope or format rules

"""

print("System prompt v2:")
print(SYSTEM_PROMPT_V2)

---
## Step 7: Evaluate v2 and Compare

In [ ]:
# Run v2 evaluation
v2_results = evaluate_retrieval(GOLDEN_SET, all_chunks, chunk_embeddings, SYSTEM_PROMPT_V2)

v2_acc = v2_results["keyword_match"].mean()
print(f"v2 Keyword Accuracy: {v2_acc:.0%} ({v2_results['keyword_match'].sum()}/{len(v2_results)})\n")

# Compare v1 vs v2
print("=" * 60)
comparison = compare_versions(v1_results, v2_results)

In [ ]:
# Inspect answers for GOLDEN_SET with (1) semantic search then (2) hybrid search

# 1) Semantic-only (uses rag_query -> search)
semantic_inspect = evaluate_retrieval(
    GOLDEN_SET, all_chunks, chunk_embeddings, SYSTEM_PROMPT_V2, top_k=3
)[["id", "difficulty", "question", "answer", "top_score", "keyword_match"]]

# 2) Hybrid (TF-IDF + semantic) -> build context -> generate
hybrid_rows = []
for qa in GOLDEN_SET:
    hyb = hybrid_search(
        qa["question"],
        chunk_embeddings, all_chunks,
        tfidf_vectorizer, tfidf_matrix,
        top_k=3, semantic_weight=0.6, keyword_weight=0.4
    )

    context = "\n\n".join(
        f"[Chunk {j+1}]\n{text}" for j, (_, _, _, _, text) in enumerate(hyb)
    )

    prompt = f"""{SYSTEM_PROMPT_V2}

<context>
{context}
</context>

Question: {qa["question"]}"""

    ans = generate(prompt, temperature=0.3, label=f"hybrid_{qa['id']}")
    hybrid_rows.append({
        "id": qa["id"],
        "difficulty": qa["difficulty"],
        "question": qa["question"],
        "answer": ans,
        "hybrid_top_score": hyb[0][1] if hyb else None,
    })

hybrid_inspect = pd.DataFrame(hybrid_rows)

# Optional: side-by-side comparison
comparison = semantic_inspect.merge(hybrid_inspect, on=["id", "difficulty", "question"], suffixes=("_semantic", "_hybrid"))
display(comparison[["id","difficulty","question","answer_semantic","answer_hybrid","top_score","hybrid_top_score","keyword_match"]])

---
## Step 8: Error Analysis

Reflect on your results. Fill in the sections below.

### Top 3 Error Patterns

**Pattern 1:** [Name]
- **What happens:** [Description]
- **Example:** [Specific question that failed]
- **Root cause:** [Why the model gets this wrong]

**Pattern 2:** [Name]
- **What happens:** [Description]
- **Example:** [Specific question that failed]
- **Root cause:** [Why the model gets this wrong]

**Pattern 3:** [Name]
- **What happens:** [Description]
- **Example:** [Specific question that failed]
- **Root cause:** [Why the model gets this wrong]

### Changes from v1 → v2

| Change | Why | Impact |
|--------|-----|--------|
| [What you modified] | [Why you expected improvement] | [Actual effect on accuracy] |
| [Second change] | [Reasoning] | [Result] |

### Remaining Risks

- **Risk 1:** [What could still go wrong]
- **Mitigation:** [How you would handle this in production]

---
## Step 9: Export Results

In [ ]:
# Export all results
# Golden set
with open("day3_lab2_golden_set.json", "w") as f:
    json.dump(GOLDEN_SET, f, indent=2)
print("Exported golden set to day3_lab2_golden_set.json")

# v1 and v2 results
v1_export = v1_results.to_dict(orient="records")
v2_export = v2_results.to_dict(orient="records")
with open("day3_lab2_results.json", "w") as f:
    json.dump({"v1": v1_export, "v2": v2_export}, f, indent=2, default=str)
print("Exported results to day3_lab2_results.json")

# Prompt log
if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day3_lab2_prompt_log.csv", index=False)
    print(f"Exported {len(PROMPT_LOG)} API calls to day3_lab2_prompt_log.csv")

---
## Checklist

- [ ] Chose a track and set `SELECTED_TRACK`
- [ ] Reviewed and understood the knowledge base documents
- [ ] Configured chunking strategy with rationale
- [ ] Chunks include metadata (chunk_id, doc_title)
- [ ] Built both dense (embedding) and sparse (TF-IDF) indices
- [ ] Tested hybrid retrieval vs semantic-only
- [ ] Tested metadata filtering
- [ ] Wrote system prompt v1 with grounding rules
- [ ] Created golden test set with 10+ questions (easy + medium + refusal)
- [ ] Ran v1 evaluation (keyword accuracy + LLM-as-judge)
- [ ] Improved prompt to v2 based on v1 errors
- [ ] Ran v2 evaluation and compared versions
- [ ] Completed error analysis (patterns, changes, risks)
- [ ] Exported all results

## Next Steps

→ Proceed to the [Day 3 Assignment](03-04_assignment_3.qmd)